# Simulation

> Virtual cameras with Gazebo/Ignition — no hardware required.

## Why Simulate?

Developing with real cameras has friction:

- You need the actual hardware
- WSL USB passthrough is fragile
- Reproducing exact conditions is hard
- CI/CD can't test against physical cameras

Simulation gives you a **virtual camera** that behaves like a real one — same topics, same messages, same API. You can develop and test your nodes without touching hardware.

## Gazebo

[Gazebo](https://gazebosim.org/) (formerly Gazebo Classic, now Gazebo Ignition/Garden) is the standard ROS2 simulator. It provides camera plugins that publish `sensor_msgs/Image` and `sensor_msgs/CameraInfo` — exactly what our nodes expect.

### Install

```sh
$ sudo apt install ros-$ROS_DISTRO-gazebo-ros-pkgs
```

### Camera Plugin

Add a camera sensor to any model in SDF:

```xml
<sensor type="camera" name="my_camera">
  <always_on>true</always_on>
  <update_rate>30</update_rate>
  <camera name="camera">
    <horizontal_fov>1.047</horizontal_fov>
    <image>
      <width>640</width>
      <height>480</height>
      <format>R8G8B8</format>
    </image>
    <clip>
      <near>0.1</near>
      <far>100</far>
    </clip>
  </camera>
  <plugin filename="libgazebo_ros_camera.so" name="camera_controller">
    <ros>
      <namespace>/camera</namespace>
    </ros>
    <frame_name>camera_link</frame_name>
  </plugin>
</sensor>
```

This publishes on:

- `/camera/image_raw` — raw images
- `/camera/camera_info` — calibration data

## Ignition Gazebo

Ignition (now Gazebo Garden/Harmonic) is the next-generation simulator. The camera plugin works similarly:

```xml
<sensor type="camera" name="camera">
  <topic>camera/image_raw</topic>
  <update_rate>30</update_rate>
  <camera>
    <horizontal_fov>1.047</horizontal_fov>
    <image>
      <width>640</width>
      <height>480</height>
      <format>R8G8B8</format>
    </image>
  </camera>
</sensor>
```

With the ROS2 bridge:

```sh
$ ros2 run ros_gz_image_bridge parameter_bridge /camera/image_raw@sensor_msgs/msg/Image
```

## Testing with Simulation

Once the simulated camera is publishing, your nodes work exactly as with real hardware:

```sh
# Terminal 1: Launch Gazebo with a world that has a camera
$ ros2 launch gazebo_ros gazebo.launch.py world:=my_world.sdf

# Terminal 2: Run your camera node (subscribes to simulated topic)
$ ros2 run ros2_cam_nbdev calib_node --ros-args \
    -p video_source:=/camera/image_raw

# Terminal 3: Verify
$ ros2 topic hz /camera/image_raw
```

> **Tip:** For headless CI testing, use `--headless` flag: `ros2 launch gazebo_ros gazebo.launch.py --headless`

## Alternatives to Gazebo

| Tool | Description | Best for |
|------|-------------|----------|
| **Gazebo Classic** | Full physics sim, mature | Complex worlds, physics testing |
| **Gazebo Ignition** | Next-gen, modular | Modern projects, better performance |
| **Webots** | Robot simulator with ROS2 integration | Multi-robot, web-based visualization |
| **AirSim** | Unreal Engine based, high-fidelity | Drone/car simulation, photorealistic |
| **rviz2** | Visualization only (no physics) | Debugging, viewing sensor data |

## Quick Test Without Gazebo

If you just need to test your node without installing Gazebo, you can publish synthetic images:

```python
# Quick script to publish test images
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Image
from cv_bridge import CvBridge
import cv2
import numpy as np

class TestPublisher(Node):
    def __init__(self):
        super().__init__('test_pub')
        self.pub = self.create_publisher(Image, '/camera/image_raw', 10)
        self.timer = self.create_timer(1.0/30, self.publish)
        self.bridge = CvBridge()
        self.i = 0

    def publish(self):
        frame = np.zeros((480, 640, 3), dtype=np.uint8)
        cv2.putText(frame, f'Frame {self.i}', (50, 240),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 3)
        msg = self.bridge.cv2_to_imgmsg(frame, 'bgr8')
        self.pub.publish(msg)
        self.i += 1

rclpy.init()
node = TestPublisher()
rclpy.spin(node)
```

This gives you a topic that your nodes can subscribe to, without any simulation framework.

## Summary

| Scenario | Approach |
|----------|----------|
| Quick unit test | Publish synthetic images with numpy/OpenCV |
| Integration test | Gazebo with camera plugin |
| High-fidelity sim | Ignition/Gazebo with realistic worlds |
| CI/CD pipeline | Headless Gazebo or synthetic publisher |